# 第五章 MiniCPM-V 公开复现入口

此快照先从[公开数据集仓库](https://modelscope.cn/datasets/pzc163/minicpm-v-inspection-part5)匿名下载固定SHA256的复现包，再执行经验证的单图与20图两模式流程。选择GPU实例与普通Python3内核，按顺序执行所有单元。数据CC BY 4.0、原创代码MIT。

原始结构结果为两模式各20/20；三图曾参与提示诊断，属于开发回归，不代表语义准确率或完整现场验收。语义问题和旧失败输出保留。


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, io, os, zipfile
PACKAGE_URL='https://modelscope.cn/datasets/pzc163/minicpm-v-inspection-part5/resolve/master/part5-reproduction-v2.zip?view=false'
PACKAGE_SHA256='55fee5d70e416e310573c08a3a2739ad4482a4f590b298b7ca307d91d15670b5'
with urlopen(PACKAGE_URL,timeout=120) as response:
    package=response.read()
assert hashlib.sha256(package).hexdigest()==PACKAGE_SHA256, 'Published package hash mismatch'
HOME=Path.cwd()/'part5-public-reproduction-v2'
HOME.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(package)) as archive:
    assert sum(x.file_size for x in archive.infolist())<5_000_000
    for item in archive.infolist():
        path=(HOME/item.filename).resolve()
        assert path.is_relative_to(HOME.resolve()) and not path.is_symlink()
        assert (item.external_attr >> 16) & 0o170000 != 0o120000
    if not (HOME/'part5-release/release-manifest.json').exists():
        archive.extractall(HOME)
os.chdir(HOME/'part5-release')
print('Anonymous download and package SHA256 verified:',len(package),PACKAGE_SHA256)


# MiniCPM-V 4.6 第五章公开复现

GPU 16x / 4x 各20图开发回归。三张图曾用于提示诊断，结构通过率不等于语义准确率。
默认 Python 3 内核即可：推理在本目录新建的虚拟环境中运行，继承魔搭基础镜像的 GPU PyTorch。
下载并解压复现包后打开本文件，从上到下执行。需要 NVIDIA GPU、可用的 CUDA PyTorch 和约4GB显存余量。
模型从官方固定版本下载（约2.5GiB），不会打包模型权重。现有缓存可通过 PART5_MODEL_DIR 指定并逐文件校验。


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, hashlib
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'release-manifest.json').is_file())
def run(args):
    subprocess.run([str(x) for x in args], cwd=ROOT, check=True)
run([sys.executable, ROOT/'examples/verify_part5_release.py'])
run([sys.executable, ROOT/'examples/bootstrap_part5.py', '--venv', ROOT/'.venv-part5',
     '--out', ROOT/'results/reproduction/environment.json'])
PY=ROOT/'.venv-part5/bin/python'


## 固定模型文件与来源
每次运行都重新核对官方文件清单与SHA256。

In [ ]:
MODEL=Path(os.environ.get('PART5_MODEL_DIR',str(ROOT/'models/minicpm-v-4.6'))).resolve()
run([PY, ROOT/'examples/download_vision_model.py', '--revision',
     'fd02313ad21ac485c595cb01231a3bded449cff5', '--out', MODEL])
model_manifest=json.loads((MODEL/'download-manifest.json').read_text())
assert model_manifest['git_commit']=='fd02313ad21ac485c595cb01231a3bded449cff5'
for entry in model_manifest['files']:
    with (MODEL/entry['path']).open('rb') as f:
        assert hashlib.file_digest(f,'sha256').hexdigest()==entry['sha256']
print('Verified model files:',len(model_manifest['files']))
DATA=ROOT/'data/inspection-validation/public-cracks/manifest.json'


## 单张图基线
保存完整原始回答，并进行严格四字段解析。

In [ ]:
sample=json.loads(DATA.read_text())['images'][0]
run([PY, ROOT/'examples/vision_baseline.py', '--model', MODEL,
     '--image', DATA.parent/sample['image'], '--out', ROOT/'results/reproduction/inspection-one.json'])


## 20图两种模式
每种模式单独加载一次模型。独立消息，无标签输入；失败原样保存。

In [ ]:
for mode in ['16x','4x']:
    run([PY, ROOT/'examples/vision_batch.py', '--model', MODEL, '--manifest', DATA,
         '--downsample', mode, '--out', ROOT/f'results/reproduction/benchmark-gpu-v2-{mode}.json'])


## 重新校验和统计
统计脚本从原始JSON重新解析，不修补回答。原始发布测量保存在results/published。

In [ ]:
run([PY, ROOT/'examples/analyze_gpu_v2.py', '--results', ROOT/'results/reproduction',
     '--manifest', DATA, '--out', ROOT/'results/reproduction'])
for mode in ['16x','4x']:
    r=json.loads((ROOT/f'results/reproduction/benchmark-gpu-v2-{mode}.json').read_text())
    print(mode, r['status'],sum(x['schema_valid'] for x in r['rows']),'/20')
print('需人工逐图核对，参阅 results/published/semantic-review.md')
